# Background

Author: Paul Inkenbrandt  
Date: March 13, 2026  
  
Notes: Found in Wellington folder that Paul shared with me

In [2]:
import calendar
import importlib
import json
import os
import pathlib
import shutil
import urllib.request
from urllib.parse import quote

import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import planetary_computer as pc
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
import rasterio
from rasterio.warp import reproject, Resampling
import requests
import statsmodels.api as sm
from pystac_client import Client
from scipy.stats import linregress
from sqlalchemy import create_engine, inspect

## Setup Database Connection

The database connection is to pull relevant station location information, and can be cirucmvented with appropriate site metadata if needed.  A VPN is needed to connect to the database using the approach below.

In [3]:
db_connection_path = "M:/My Drive/Python/dbconnect.py"

spec = importlib.util.spec_from_file_location("dbconnect", db_connection_path)
dbconnect =importlib.util.module_from_spec(spec)
spec.loader.exec_module(dbconnect)

engine = dbconnect.postconn_ugs_write()

In [7]:
query = "SELECT stationid, latitude, longitude FROM groundwater.station_attribute_view"
station_locations = pd.read_sql(query, engine).dropna()
station_locations.set_index('stationid', inplace=True)
print(station_locations)

            latitude    longitude
stationid                        
US-CdM     37.524693  -109.745804
US-UTB       40.7847    -113.8298
US-UTD       38.1038    -109.5952
US-UTE     37.735287  -111.570816
US-UTG     39.018806  -110.162766
US-UTJ     37.281746   -109.53489
US-UTL     40.187252   -109.65754
US-UTM      38.58785   -109.57749
US-UTN     39.689617   -111.88127
US-UTP     41.060347  -112.101358
US-UTV      40.12641  -109.954735
US-UTW     39.445362   -110.72905


In [21]:
def get_landsat_data(latitude: float, longitude: float, BEG_DATE = "2020-01-01", END_DATE = "2026-09-25", verbose: bool = False):
    """Performs a search for Landsat data using the Planetary Computer STAC API and retrieves metadata for the specified location and date range.

    Args:
        latitude (float): The latitude of the location.
        longitude (float): The longitude of the location.
        BEG_DATE (str, optional): The beginning date for the search. Defaults to "2020-01-01".
        END_DATE (str, optional): The end date for the search. Defaults to "2026-09-25".
        verbose (bool, optional): If True, prints additional information. Defaults to False.

    Returns:
        _type_: _description_
    """
    cat = Client.open("https://planetarycomputer.microsoft.com/api/stac/v1",
                    modifier=pc.sign_inplace)
    items = cat.search(collections=["landsat-c2-l2"],
                    intersects={"type": "Point", "coordinates": [longitude, latitude]},
                    datetime=f"{pd.to_datetime(BEG_DATE):%Y-%m-%d}/{pd.to_datetime(END_DATE):%Y-%m-%d}").item_collection()

    # quick table of the item-level metadata
    df = pd.DataFrame([{"id": i.id, **i.properties} for i in items])
    df["datetime"] = pd.to_datetime(df["datetime"])
    if verbose:
        pass
    else:
        df = df[['datetime', 
                 'eo:cloud_cover', 
                 'landsat:scene_id',
                 'platform', 
                 'instruments',
                 'view:sun_azimuth',
                 'view:sun_elevation'
                 ]]
    # full MTL for one scene
    #mtl = json.load(urllib.request.urlopen(items[0].assets["mtl.json"].href))
    #attrs = mtl["LANDSAT_METADATA_FILE"]["IMAGE_ATTRIBUTES"]
    return df

landsat_data = {}

for station_id, row in station_locations.iterrows():
    latitude = row['latitude']
    longitude = row['longitude']
    landsat_df = get_landsat_data(latitude, longitude)
    print(f"Station ID: {station_id}")
    landsat_data[station_id] = landsat_df

landsat_passes = pd.concat(landsat_data) 

Station ID: US-CdM
Station ID: US-UTB
Station ID: US-UTD
Station ID: US-UTE
Station ID: US-UTG
Station ID: US-UTJ
Station ID: US-UTL
Station ID: US-UTM
Station ID: US-UTN
Station ID: US-UTP
Station ID: US-UTV
Station ID: US-UTW


In [22]:
landsat_passes.to_parquet('M:/Shared drives/UGS_Flux/Data_Processing/landsat_overpass_dates.parquet', index=True)

# Inputs

In [23]:
MAIN_DIR = pathlib.Path("D:/footprints")
# ── Inputs ────────────────────────────────────────────────────────────────
FFP_DIR    = pathlib.Path("D:/footprints")
OPENET_DIR = pathlib.Path("M:/My Drive/OpenET_Daily_Tiffs/")

date = "20230715"  # example

footprint_path = FFP_DIR / f"ffp_UTW_daily_etw_{date}.tif"
openet_path    = OPENET_DIR / f"ensemble_et_{date}.tif"

US-CdM US-UTJ_US-CdM
Model: eemetric
Model: ssebop
Model: ensemble
Model: sims
Model: geesebal
Model: disalexi
Model: ptjpl
Model: desktop.ini
US-UTB US-UTB
Model: eemetric
Model: ensemble
Model: desktop.ini
US-UTD US-UTD_US-UTG_US-UTM_US-UTW
Model: eemetric
Model: ensemble
Model: ptjpl
Model: ssebop
Model: disalexi
Model: geesebal
Model: sims
Model: desktop.ini
US-UTE US-UTE
Model: ensemble
Model: eemetric
Model: sims
Model: ssebop
Model: disalexi
Model: geesebal
Model: ptjpl
Model: desktop.ini
US-UTG US-UTD_US-UTG_US-UTM_US-UTW
Model: eemetric
Model: ensemble
Model: ptjpl
Model: ssebop
Model: disalexi
Model: geesebal
Model: sims
Model: desktop.ini
US-UTJ US-UTJ_US-CdM
Model: eemetric
Model: ssebop
Model: ensemble
Model: sims
Model: geesebal
Model: disalexi
Model: ptjpl
Model: desktop.ini
US-UTL US-UTL_US-UTV
Model: ptjpl
Model: geesebal
Model: disalexi
Model: sims
Model: ssebop
Model: eemetric
Model: ensemble
Model: desktop.ini
US-UTM US-UTD_US-UTG_US-UTM_US-UTW
Model: eemetric
Model

In [ ]:
# ── Step 1: Load the footprint (EPSG:32612, 10 m, sums to 1.0) ───────────
def load_footprint(footprint_path):
    """Load the footprint raster and return the footprint array, transform, CRS, height, and width.

    Args:
        footprint_path (str): Path to the footprint raster file.

    Returns:
        tuple: A tuple containing the footprint array, transform, CRS, height, and width.
    """

    
    with rasterio.open(footprint_path) as src:
        phi          = src.read(1).astype(np.float64)   # shape (101, 101)
        fp_transform = src.transform
        fp_crs       = src.crs
        fp_height    = src.height
        fp_width     = src.width

    # Confirm it sums to ~1.0 (dimensionless spatial weight, not a density)
    return phi, fp_transform, fp_crs, fp_height, fp_width

# ── Step 2: Load and reproject OpenET onto the footprint grid ─────────────
# OpenET is EPSG:4326, ~30 m — needs reprojection to match footprint grid

def load_and_reproject_openet(openet_path, fp_transform, fp_crs, fp_height, fp_width):
    """Load the OpenET raster, reproject it onto the footprint grid, and return the reprojected array.

    Args:
        openet_path (str): Path to the OpenET raster file.
        fp_transform (Affine): Affine transform of the footprint grid.
        fp_crs (CRS): Coordinate reference system of the footprint grid.
        fp_height (int): Height of the footprint grid.
        fp_width (int): Width of the footprint grid.

    Returns:
        np.ndarray: The reprojected OpenET array.
    """
    with rasterio.open(openet_path) as src:
        openet_raw     = src.read(1).astype(np.float64)
        src_transform  = src.transform
        src_crs        = src.crs
        nodata         = src.nodata

    if nodata is not None:
        openet_raw[openet_raw == nodata] = np.nan

    openet_reproj = np.full((fp_height, fp_width), np.nan, dtype=np.float64)
    reproject(
        source        = openet_raw,
        destination   = openet_reproj,
        src_transform = src_transform,
        src_crs       = src_crs,
        dst_transform = fp_transform,
        dst_crs       = fp_crs,
        resampling    = Resampling.bilinear,
        src_nodata    = np.nan,
        dst_nodata    = np.nan,
    )
    
    return openet_reproj


# ── Step 3: Footprint-weighted ET ─────────────────────────────────────────
def compute_footprint_weighted_et(phi, openet_reproj):
    """Compute the footprint-weighted ET using the footprint and reprojected OpenET arrays.

    Args:
        phi (np.ndarray): Footprint array.
        openet_reproj (np.ndarray): Reprojected OpenET array.

    Returns:
        float: The footprint-weighted ET in mm/day.
    """

    # Only use pixels where footprint > 0 AND OpenET is valid
    valid = (phi > 0) & np.isfinite(openet_reproj)

    # Renormalize footprint over valid pixels only
    # (handles any edge pixels that fell outside the OpenET extent)
    phi_valid = phi.copy()
    phi_valid[~valid] = 0.0
    phi_norm = phi_valid / phi_valid.sum()   # still sums to 1.0 over valid pixels

    # Weighted average:  ET_fp = Σ φ(x,y) · OpenET(x,y)
    # Because Σφ = 1.0, this is a weighted mean in mm/day — no pixel area needed
    ET_fp = float(np.sum(phi_norm * openet_reproj * valid))

    # For reference: simple unweighted spatial mean
    ET_mean = float(np.nanmean(openet_reproj[valid]))

    return ET_fp, ET_mean

In [35]:
footprint_path = "D:/footprints/US-UTJ/ffp_US-UTJ_daily_etw_20240606.tif"

def load_footprint(footprint_path):
    """Load the footprint raster and return the footprint array, transform, CRS, height, and width.

    Args:
        footprint_path (str): Path to the footprint raster file.

    Returns:
        tuple: A tuple containing the footprint array, transform, CRS, height, and width.
    """

    # ── Step 1: Load the footprint (EPSG:32612, 10 m, sums to 1.0) ───────────
    with rasterio.open(footprint_path) as src:
        phi          = src.read(1).astype(np.float64)   # shape (101, 101)
        fp_transform = src.transform
        fp_crs       = src.crs
        fp_height    = src.height
        fp_width     = src.width

    # Confirm it sums to ~1.0 (dimensionless spatial weight, not a density)
    return phi, fp_transform, fp_crs, fp_height, fp_width

phi, fp_transform, fp_crs, fp_height, fp_width = load_footprint(footprint_path)

# Step 2: Load and reproject OpenET onto the footprint grid

In [39]:
# ── Step 2: Load and reproject OpenET onto the footprint grid ─────────────
# OpenET is EPSG:4326, ~30 m — needs reprojection to match footprint grid

def load_and_reproject_openet(openet_path, fp_transform, fp_crs, fp_height, fp_width):
    """Load the OpenET raster, reproject it onto the footprint grid, and return the reprojected array.

    Args:
        openet_path (str): Path to the OpenET raster file.
        fp_transform (Affine): Affine transform of the footprint grid.
        fp_crs (CRS): Coordinate reference system of the footprint grid.
        fp_height (int): Height of the footprint grid.
        fp_width (int): Width of the footprint grid.

    Returns:
        np.ndarray: The reprojected OpenET array.
    """
    with rasterio.open(openet_path) as src:
        openet_raw     = src.read(1).astype(np.float64)
        src_transform  = src.transform
        src_crs        = src.crs
        nodata         = src.nodata

    if nodata is not None:
        openet_raw[openet_raw == nodata] = np.nan

    openet_reproj = np.full((fp_height, fp_width), np.nan, dtype=np.float64)
    reproject(
        source        = openet_raw,
        destination   = openet_reproj,
        src_transform = src_transform,
        src_crs       = src_crs,
        dst_transform = fp_transform,
        dst_crs       = fp_crs,
        resampling    = Resampling.bilinear,
        src_nodata    = np.nan,
        dst_nodata    = np.nan,
    )
    
    return openet_reproj

openet_path = "M:/My Drive/OpenET_Daily_Tiffs/US-UTJ_US-CdM/ensemble/US-CdM_US-UTJ_ensemble_et_2024_06_06_11c2e.tif"

openet_reproj = load_and_reproject_openet(openet_path, fp_transform, fp_crs, fp_height, fp_width)

# Step 3: Footprint-weighted ET

In [58]:
def compute_footprint_weighted_et(phi, openet_reproj):
    """Compute the footprint-weighted ET using the footprint and reprojected OpenET arrays.

    Args:
        phi (np.ndarray): Footprint array.
        openet_reproj (np.ndarray): Reprojected OpenET array.

    Returns:
        float: The footprint-weighted ET in mm/day.
    """
    # ── Step 3: Footprint-weighted ET ─────────────────────────────────────────
    # Only use pixels where footprint > 0 AND OpenET is valid
    valid = (phi > 0) & np.isfinite(openet_reproj)

    # Renormalize footprint over valid pixels only
    # (handles any edge pixels that fell outside the OpenET extent)
    phi_valid = phi.copy()
    phi_valid[~valid] = 0.0
    phi_norm = phi_valid / phi_valid.sum()   # still sums to 1.0 over valid pixels

    # Weighted average:  ET_fp = Σ φ(x,y) · OpenET(x,y)
    # Because Σφ = 1.0, this is a weighted mean in mm/day — no pixel area needed
    ET_fp = float(np.sum(phi_norm * openet_reproj * valid))

    # For reference: simple unweighted spatial mean
    ET_mean = float(np.nanmean(openet_reproj[valid]))

    return ET_fp, ET_mean

ET_fp, ET_mean = compute_footprint_weighted_et(phi, openet_reproj)
print(f"Footprint-weighted OpenET: {ET_fp:.3f} mm/day")
print(f"Unweighted spatial mean:   {ET_mean:.3f} mm/day")

Footprint-weighted OpenET: 0.629 mm/day
Unweighted spatial mean:   0.719 mm/day


In [59]:
models = ['ensemble', 'disalexi', 'eemetric', 'ensemble', 'geesebal', 'ptjpl', 'ssebop', 'sims']

station_oet = {}
oet = {}

FFP_DIR    = pathlib.Path("D:/footprints")
OPENET_DIR = pathlib.Path("M:/My Drive/OpenET_Daily_Tiffs/")

for station_id, row in station_locations.iterrows():
    footprint_dir = FFP_DIR / station_id
    print(f"Station ID: {station_id}")
    for file in footprint_dir.glob('*daily*.tif'):
        filename = str(file.stem)
        filename_parts = filename.split('_')
        footprint_date = pd.to_datetime(filename_parts[-1], format='%Y%m%d')
        search_str = f"*{station_id}*{footprint_date:%Y_%m_%d}*.tif"
        for oetfile in OPENET_DIR.rglob(search_str):
            if oetfile.is_file():
                oetfilename = str(oetfile.name)
                for model in models:
                    if model in oetfilename:
                        
                        phi, fp_transform, fp_crs, fp_height, fp_width =load_footprint(file)
                        openet_reproj = load_and_reproject_openet(openet_path, fp_transform, fp_crs, fp_height, fp_width)
                        ET_fp, ET_mean = compute_footprint_weighted_et(phi, openet_reproj)
                        oet[f"{footprint_date:%Y%m%d}_{model}"] = pd.Series({'station_id': station_id, 
                               'model': model, 
                               'footprint_file': file.stem, 
                               'openet_file': oetfilename, 
                               'footprint_date': footprint_date, 
                               'ET_fp': ET_fp, 
                               'ET_mean': ET_mean,
                               'unity_check': phi.sum()
                               })
                        print(f"          Model: {model}, OpenET file: {oetfilename}, Date: {footprint_date: %Y-%m-%d}")
    station_oet[station_id] = pd.concat(oet)
    station_oet[station_id].to_parquet(f"station_oet_{station_id}.parquet")
    print(f"Station {station_id} complete.")
    #    if station_id in filename:
    #       print(station_id, filename)
    #       for model in file.iterdir():
    #           print(f"Model: {model.name}")

Station ID: US-CdM
          Model: eemetric, OpenET file: US-CdM_US-UTJ_eemetric_et_2023_10_25_5bb14.tif, Date:  2023-10-25
          Model: ssebop, OpenET file: US-CdM_US-UTJ_ssebop_et_2023_10_25_79dea.tif, Date:  2023-10-25
          Model: ensemble, OpenET file: US-CdM_US-UTJ_ensemble_et_2023_10_25_45c32.tif, Date:  2023-10-25
          Model: ensemble, OpenET file: US-CdM_US-UTJ_ensemble_et_2023_10_25_45c32.tif, Date:  2023-10-25
          Model: eemetric, OpenET file: US-CdM_US-UTJ_eemetric_et_2023_10_26_8059b.tif, Date:  2023-10-26
          Model: ssebop, OpenET file: US-CdM_US-UTJ_ssebop_et_2023_10_26_7ae33.tif, Date:  2023-10-26
          Model: ensemble, OpenET file: US-CdM_US-UTJ_ensemble_et_2023_10_26_8e3ca.tif, Date:  2023-10-26
          Model: ensemble, OpenET file: US-CdM_US-UTJ_ensemble_et_2023_10_26_8e3ca.tif, Date:  2023-10-26
          Model: eemetric, OpenET file: US-CdM_US-UTJ_eemetric_et_2023_10_27_5ca3e.tif, Date:  2023-10-27
          Model: ssebop, OpenET fil

KeyboardInterrupt: 

In [60]:
from fp_weighted_openet import run
results = run(station_locations.index, "D:/footprints", "M:/My Drive/OpenET_Daily_Tiffs", out_dir=".")

index_openet: skipped 1462 duplicate station/date/model files
Indexed 24718 OpenET files in 3.0 s
Station US-CdM: 703 days, 3184 rows in 175.2 s
Station US-UTB: 0 days, 0 rows in 0.0 s
Station US-UTD: 831 days, 3670 rows in 997.0 s
Station US-UTE: 463 days, 2477 rows in 184.0 s
Station US-UTG: 0 days, 0 rows in 0.0 s
Station US-UTJ: 0 days, 0 rows in 0.0 s
Station US-UTL: 283 days, 1981 rows in 150.5 s
Station US-UTM: 0 days, 0 rows in 0.0 s
Station US-UTN: 0 days, 0 rows in 0.0 s
Station US-UTP: 0 days, 0 rows in 0.0 s
Station US-UTV: 0 days, 0 rows in 0.0 s
Station US-UTW: 0 days, 0 rows in 0.0 s
